In [ ]:
import os
import sys
from pathlib import Path

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

torch.manual_seed(8008135)

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("CODE_DIR:", CODE_DIR)
print("CODE_DIR contents:", os.listdir(CODE_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device set to {device}")

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

In [ ]:
from Sudoku.sudoku import print_sudoku_comparison

from Datasets.Sudoku_DataLoader import get_loaders

from BiLSTM_Model.BiLSTM_Sudoku import SudokuBiLSTM, BiLSTMConfig
from BiLSTM_Model.BiLSTM_Train import train_bilstm

from Utils.schedules import cosine_schedule_with_warmup_lr_lambda
from Utils.checkpointing import load_checkpoint
from Utils.visualization import show_sudoku_predictions

In [ ]:
train_size = 2**13
test_size = 2**8
batch_size = 2**8

train_loader, val_loader = get_loaders(
    train_size=train_size,
    test_size=test_size,
    batch_size=batch_size,
)

In [ ]:
model_config = BiLSTMConfig(
    vocab_size=10,
    num_classes=10,
    embedding_dim=256,
    hidden_size=560,
    num_layers=5,
    dropout=0.10,
)

lr = 3e-4
beta1 = 0.9
beta2 = 0.95
weight_decay = 0.1
num_epochs = 10

checkpoint_dir = "checkpoints"

In [ ]:
model = SudokuBiLSTM(model_config).to(device)

print(
    "Number of trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

num_training_steps = len(train_loader) * num_epochs
num_warmup_steps = int(0.05 * num_training_steps)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_schedule_with_warmup_lr_lambda(
        step,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        min_ratio=0.1,
    ),
)

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)

In [ ]:
model, best_metric = train_bilstm(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=5,
    validate_every=5,
    val_loader=val_loader
)

model.eval()

print("Best board accuracy used for checkpointing:", best_metric)

In [ ]:
show_sudoku_predictions(
    model,
    val_loader,
    device,
    print_sudoku_comparison,
    num_examples=10,
)

In [ ]:
torch.save(
    model.state_dict(),
    "checkpoints/bilstm_final_state_dict.pt",
)